# 00 - Gaussian Process Regression Fundamentals

The goal of this notebook is to understand `GaussianProcessRegressor` **before** using it inside Bayesian Optimization.

The key distinction is:

- **Gaussian Process Regression (GPR):** a probabilistic regression and uncertainty model.
- **Bayesian Optimization:** an optimization strategy that uses a surrogate model such as GPR to decide where to evaluate an expensive objective next.

Therefore, `sklearn.gaussian_process.GaussianProcessRegressor` is not a black-box optimizer by itself.

## 1. What does a Gaussian Process predict?

For an input point \(x\), a Gaussian Process can provide both a predictive mean

\[\mu(x)\]

and predictive uncertainty

\[\sigma(x).\]

This uncertainty is central to Bayesian Optimization. The optimizer can consider both where the objective is predicted to be promising and where the model is still uncertain.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern

## 2. A small teaching function

The following function is not a manufacturing law. It is only a simple nonlinear response used to visualize how the GP interpolates sparse observations and represents uncertainty between them.

In [ ]:
def true_function(x):
    return np.sin(1.5 * x) + 0.15 * x

X_train = np.array([[-2.5], [-1.0], [0.2], [1.8], [3.0]])
y_train = true_function(X_train[:, 0])

X_grid = np.linspace(-3.0, 3.5, 400).reshape(-1, 1)

## 3. Kernel selection

A kernel expresses assumptions about similarity and smoothness. Here we use a Matérn kernel with `nu=2.5`.

A common interpretation is:

- `nu=1.5`: functions are approximately once differentiable,
- `nu=2.5`: functions are approximately twice differentiable.

Smaller values allow rougher functions; large `nu` values approach RBF-like smoothness.

In [ ]:
kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(
    length_scale=1.0,
    length_scale_bounds=(1e-3, 1e3),
    nu=2.5,
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-8,
    normalize_y=True,
    n_restarts_optimizer=5,
    random_state=42,
)

gp.fit(X_train, y_train)
mean, std = gp.predict(X_grid, return_std=True)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(X_grid[:, 0], true_function(X_grid[:, 0]), label="True function")
plt.plot(X_grid[:, 0], mean, label="GP predictive mean")
plt.fill_between(
    X_grid[:, 0],
    mean - 1.96 * std,
    mean + 1.96 * std,
    alpha=0.2,
    label="Approximate 95% uncertainty band",
)
plt.scatter(X_train[:, 0], y_train, label="Observations")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Gaussian Process Regression")
plt.legend()
plt.grid(True)
plt.show()

## 4. How to read the figure

Uncertainty generally decreases near observed points and increases farther away. Bayesian Optimization uses this structure to balance:

- **exploitation:** sample where a good objective value is predicted,
- **exploration:** sample where uncertainty is high enough to make a new observation informative.

This is why probabilistic surrogate models are useful when each real experiment is expensive.

## 5. `alpha` versus `WhiteKernel`

These concepts should not be treated as identical.

`alpha` is added to the diagonal of the kernel matrix. It can improve numerical stability or represent known observation-noise variance.

`WhiteKernel` is an explicit noise component in the kernel and can allow the noise level to be learned from data.

Using both can be valid, but their roles should be stated explicitly. Educational examples are often clearer when unnecessary double noise modeling is avoided.

## 6. Limitations of standard Gaussian Processes

The core linear algebra of a classical exact GP scales approximately as \(O(n^3)\) in the number of observations. GP models are therefore especially attractive when data are limited, each new observation is expensive, and uncertainty matters.

They are usually not the first choice for millions of cheap training observations.

## 7. Next notebook

The next notebook uses this GP as a surrogate model and adds acquisition functions plus sequential sampling to form a complete Bayesian Optimization algorithm.